# README Benchmark Runner

Run the cells top to bottom. Edit only the configuration cell for normal experiment changes.

## Configuration

In [ ]:
EXPERIMENT = {
    "run_name": "default_run",
    "data_csv": "data/repo_list.csv",
    "tools": {
        "osa": {"enabled": True},
        "readmeready": {"enabled": True},
        "larch": {"enabled": True, "mode": "local"},
    },
    "models": [
        "openai/gpt-4.1",
        "anthropic/claude-sonnet-4",
        "google/gemma-3-27b-it",
    ],
    "judge": {
        "api": "openrouter",
        "base_url": "https://openrouter.ai/api/v1",
        "model": "gpt-4.1",
    },
    "reset": {
        "repos": False,
        "tool_outputs": False,
        "evaluation": False,
    },
    # Set to an integer for smoke tests, or None for the full CSV.
    "limit_repos": None,
}


## Imports And Paths

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
candidates = [cwd, cwd / "benchmark_README", *cwd.parents]
BM = next((p for p in candidates if (p / "src" / "notebook_utils.py").is_file()), cwd)
if str(BM) not in sys.path:
    sys.path.insert(0, str(BM))

from src.notebook_utils import build_paths, ensure_directories, load_env_file, read_repo_table, prepare_repositories
from src.tool_runners import run_selected_tools
from src.evaluation import evaluate_outputs

load_env_file(BM / ".env")
paths = build_paths(EXPERIMENT["run_name"])
ensure_directories(paths)
print(paths)


## Load Repository Table

In [ ]:
repo_df = read_repo_table(EXPERIMENT["data_csv"])
if EXPERIMENT.get("limit_repos"):
    repo_df = repo_df.head(int(EXPERIMENT["limit_repos"]))
repo_df.head()


## Pre-flight: Clone, Checkout, Snapshot

In [ ]:
preflight = prepare_repositories(
    repo_df,
    paths,
    reset=bool(EXPERIMENT.get("reset", {}).get("repos", False)),
)
preflight


## Run Selected Tools

In [ ]:
tool_status = run_selected_tools(repo_df, paths, EXPERIMENT)
tool_status.tail(20)


## Evaluate Generated READMEs

In [ ]:
eval_rows = evaluate_outputs(
    paths,
    EXPERIMENT,
    reset=bool(EXPERIMENT.get("reset", {}).get("evaluation", False)),
)
eval_rows.tail(20)


## Summary

In [ ]:
summary_path = paths.evaluation_dir / "final_summary.csv"
if summary_path.is_file():
    import pandas as pd
    display(pd.read_csv(summary_path))
else:
    print("No summary yet. Run evaluation after at least one successful tool output.")
